# Macro-Credit Risk Stress Testing Engine
## Cuaderno 02: Modelamiento Econométrico (SARIMAX vs VAR), Diagnósticos y Stress Testing
**Objetivo:** Calibrar los modelos SARIMAX y VAR, validar los supuestos regulatorios de residuos, ejecutar backtesting temporal (rolling origin) con función de pérdida asimétrica y proyectar el impacto en balance bajo escenarios macroeconómicos.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.models import SARIMAXModelWrapper, VARModelWrapper, TimeDimensionBacktester
from src.stress_engine import StressTester, DEFAULT_SCENARIOS

### 1. Carga de Variables Estacionarias Rezagadas

In [ ]:
proc_path = PROJECT_ROOT / 'data/processed/macro_npl_processed.csv'
df = pd.read_csv(proc_path, index_col='date', parse_dates=True)
target = df['npl_diff']
exog_cols = [c for c in df.columns if c not in ['npl_diff', 'npl_level']]
exog = df[exog_cols]
print(f'Dataset para estimación: {len(df)} observaciones')
df.head()

### 2. Estimación y Diagnósticos Regulatorios de SARIMAX

In [ ]:
sarimax = SARIMAXModelWrapper(order=(1, 0, 1), seasonal_order=(1, 0, 0, 12))
sarimax.fit(target, exog=exog)

print('--- TABLA DE COEFICIENTES ---')
print(sarimax.get_coefficients_table().to_string())

print('\n--- DIAGNÓSTICOS DE RESIDUOS (LJUNG-BOX / JARQUE-BERA / ARCH-LM) ---')
diag = sarimax.run_diagnostics()
print(diag.summary_df.to_string(index=False))

### 3. Modelo de Vectores Autorregresivos (VAR): Estabilidad e IRF

In [ ]:
var_cols = ['gdp_growth_yoy_lag3', 'policy_rate_diff_lag8', 'inflation_diff_lag9', 'fx_depreciation_mom_lag12', 'npl_diff']
var_model = VARModelWrapper(target_col='npl_diff', maxlags=3)
var_model.fit(df[var_cols], ic='aic')

is_stable, max_root = var_model.check_dynamic_stability()
print(f'Estabilidad Dinámica: {is_stable} | Max Eigenvalue: {max_root:.4f} < 1.0')

gc = var_model.test_granger_causality()
print('Causalidad de Granger p-values:', gc)

### 4. Backtesting Temporal Expansivo con Pérdida Asimétrica

In [ ]:
backtester = TimeDimensionBacktester(min_train_periods=72, forecast_horizon=12, step_size=6)
comp_df = backtester.evaluate_models(target, exog, df[var_cols])
print(comp_df.to_string(index=False))

### 5. Motor de Stress Testing: Simulación de Escenarios e Impacto NIIF 9

In [ ]:
tester = StressTester(processed_data_path=proc_path, raw_data_path=PROJECT_ROOT / 'data/raw/macro_npl_raw.csv')
results = tester.run_all_scenarios(portfolio_size_mm=5000.0, lgd_pct=75.0, save_results=True)

impact_summary = pd.DataFrame(results['impact_summary'])
cols_show = ['scenario_name', 'projected_npl_month12_pct', 'delta_npl_bps', 'additional_provisions_required_mm', 'total_provisions_stressed_mm', 'capital_cet1_impact_bps']
print(impact_summary[cols_show].to_string(index=False))